## Demand Planning with Open Orders (Forecast Minimum Constraints)

This notebook extends the [demand planning use case](../use_cases/demand_planning/demand_planning.ipynb) by incorporating open orders as minimum forecast constraints. Open orders represent committed customer orders, contractual obligations, or inventory requirements that forecasts must respect.

For details on the general demand planning workflow and configuration options, refer to the [demand planning notebook](../use_cases/demand_planning/demand_planning.ipynb).

In [1]:
import time

from futureexpert import (DataDefinition,
                          ExpertClient,
                          FileSpecification,
                          ForecastingConfig,
                          MethodSelectionConfig,
                          PreprocessingConfig,
                          ReportConfig,
                          TsCreationConfig)
import futureexpert.checkin as checkin

client = ExpertClient()

INFO:futureexpert.expert_client:Successfully logged in for group group-expert.


### CHECK-IN Actuals

We check in the demand data at the material level, using the same configuration as the demand planning use case.

In [2]:
actuals_version_id = client.check_in_time_series(
    raw_data_source='../use_cases/demand_planning/demo_demand_planning_data.csv',
    file_specification=FileSpecification(delimiter=';', decimal='.'),
    data_definition=DataDefinition(
        date_column=checkin.DateColumn(name='DATE', format='%d.%m.%Y', name_new='Date'),
        value_columns=[checkin.ValueColumn(name='DEMAND', name_new='Demand')],
        group_columns=[
            checkin.GroupColumn(name='CUSTOMER', name_new='Customer'),
            checkin.GroupColumn(name='MATERIAL', name_new='Material'),
            checkin.GroupColumn(name='REGION', name_new='Region')
        ]
    ),
    config_ts_creation=TsCreationConfig(
        time_granularity='monthly',
        start_date='2007-10-01',
        end_date='2024-06-01',
        value_columns_to_save=['Demand'],
        grouping_level=['Material'],
        missing_value_handler='setToZero',
        filter=[checkin.FilterSettings(type='exclusion', variable='Material', items=['C011414576'])]
    )
)

INFO:futureexpert.expert_client:Transforming input data...
INFO:futureexpert.expert_client:Creating time series using CHECK-IN...
INFO:futureexpert.expert_client:Finished time series creation.


### CHECK-IN Open Orders

Open orders are checked in as a separate time series version. The grouping columns must match the actuals (Customer, Material, Region) so that minimums can be mapped to the correct forecasts.  The name of the open order time series is not used for mapping to actuals. The `grouping_level` is set to Material to match the actuals aggregation level. Open order dates must fall within the forecast horizon to be respected in forecasts.

In [3]:
open_orders_version_id = client.check_in_time_series(
    raw_data_source='../example_data/open_orders_demand.csv',
    file_specification=FileSpecification(delimiter=';', decimal='.'),
    data_definition=DataDefinition(
        date_column=checkin.DateColumn(name='Date', format='%d.%m.%Y'),
        value_columns=[checkin.ValueColumn(name='OpenOrder')],
        group_columns=[
            checkin.GroupColumn(name='Customer'),
            checkin.GroupColumn(name='Material'),
            checkin.GroupColumn(name='Region')
        ]
    ),
    config_ts_creation=TsCreationConfig(
        time_granularity='monthly',
        start_date='2024-07-01',
        value_columns_to_save=['OpenOrder'],
        grouping_level=['Material'],
        missing_value_handler='keepNaN'
    )
)

INFO:futureexpert.expert_client:Transforming input data...
INFO:futureexpert.expert_client:Creating time series using CHECK-IN...
INFO:futureexpert.expert_client:Finished time series creation.


In [4]:
open_orders = client.get_time_series(open_orders_version_id).time_series
open_order_materials = [ts.grouping['Material'] for ts in open_orders]

### FORECAST with Open Orders

We use the same forecast configuration as the demand planning use case, with the addition of `forecast_minimum_version` pointing to the open orders. This ensures forecasts respect the minimum constraints. We only forecast the materials for which open orders are available.

In [5]:

fc_report_config = ReportConfig(
    title='Monthly Demand Forecast with Open Orders',
    actuals_filter={'grouping.Material': {'$in': open_order_materials}},
    preprocessing=PreprocessingConfig(
        remove_leading_zeros=True,
        detect_outliers=True,
        replace_outliers=True,
        detect_changepoints=True,
        detect_quantization=True,
        phase_out_method='AUTO_FEW_OBS'
    ),
    forecasting=ForecastingConfig(
        fc_horizon=18,
        lower_bound=0,
        use_ensemble=True,
        confidence_level=0.75,
        round_forecast_to_integer=True,
        forecast_minimum_version=open_orders_version_id
    ),
    method_selection=MethodSelectionConfig(
        number_iterations=18,
        refit=True,
        step_weights={3: 1, 4: 1, 5: 0.5, 6: 0.5},
        additional_accuracy_measures=['me', 'mae'],
        phase_out_fc_methods=['ZeroForecast']
    ),
    max_ts_len=72
)

forecast_identifier = client.start_forecast(version=actuals_version_id, config=fc_report_config)


INFO:futureexpert.expert_client:Preparing data for forecast...
INFO:futureexpert.expert_client:Finished data preparation for forecast.
INFO:futureexpert.expert_client:Started creating forecasting report with FORECAST...
INFO:futureexpert.expert_client:Report created with ID 170738. Forecasts are running...


In [6]:
while not (current_status := client.get_report_status(id=forecast_identifier)).is_finished:
    time.sleep(10)

results = client.get_fc_results(id=forecast_identifier, include_backtesting=True, include_k_best_models=3)

### Verify Open Order Minimums

We verify that forecasts for materials with open orders respect the specified minimum values.

In [7]:
import numpy as np

for open_order_ts in open_orders:
    forecast_result = results.get_forecast_result(open_order_ts.name.replace('OpenOrder', 'Demand'))
    best_model = forecast_result.best_model
    print(f'{forecast_result.input.actuals.name} ({best_model.model_name}):')
    for expected_minimum in open_order_ts.values:
        point_forecast_value = next(fc_value.point_forecast_value for fc_value in best_model.forecasts
                                    if fc_value.time_stamp_utc == expected_minimum.time_stamp_utc)
        raw_model_forecast_value = next(fc_value.point_forecast_value for fc_value in best_model.raw_model_forecasts
                                        if fc_value.time_stamp_utc == expected_minimum.time_stamp_utc)
        status = 'OK' if point_forecast_value >= expected_minimum.value else 'VIOLATED'
        print(f'  {expected_minimum.time_stamp_utc.strftime("%Y-%m")}: Forecast={point_forecast_value:.0f}, Minimum={expected_minimum.value}, Raw Model Forecast: {raw_model_forecast_value}, {status}')

    raw_model_fc_values = np.array([forecast.point_forecast_value for forecast in best_model.raw_model_forecasts])
    final_fc_values = np.array([forecast.point_forecast_value for forecast in best_model.forecasts])
    if (num_differing_forecast_values := (~np.isclose(final_fc_values, raw_model_fc_values)).sum()) > 0:
        print(f'  {num_differing_forecast_values} point forecast values have been adapted to the given open orders')
        assert len(best_model.changed_forecast_values) == num_differing_forecast_values

Demand-C048529686 (MedianAS):
  2024-08: Forecast=2820, Minimum=250.0, Raw Model Forecast: 2820.0, OK
Demand-C011414575 (Ensemble):
  2024-07: Forecast=500, Minimum=500.0, Raw Model Forecast: 234.0, OK
  2024-08: Forecast=450, Minimum=450.0, Raw Model Forecast: 312.0, OK
  2024-10: Forecast=380, Minimum=380.0, Raw Model Forecast: 234.0, OK
  3 point forecast values have been adapted to the given open orders
Demand-C024897037 (AutoEsCov):
  2024-07: Forecast=336, Minimum=300.0, Raw Model Forecast: 336.0, OK
  2024-09: Forecast=270, Minimum=200.0, Raw Model Forecast: 270.0, OK
